In [1]:
import pandas as pd
import numpy as np
from imblearn.under_sampling import RandomUnderSampler
import os
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger()

# Constants
DATA_FILE = 'full_data_unhealthy_imputed.csv'
OUTPUT_DIR = 'balanced_datasets'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def load_and_preprocess():
    """Load and preprocess data with enhanced validation and reporting"""
    logger.info(f"Loading data from {DATA_FILE}")
    try:
        data = pd.read_csv(DATA_FILE)
        
        # Validate required columns
        required_cols = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'hour', 'date', 
                         'oestrus', 'calving', 'lameness', 'mastitis', 'cow']
        missing_cols = set(required_cols) - set(data.columns)
        if missing_cols:
            raise ValueError(f"Missing required columns: {missing_cols}")
        
        # Report original disease prevalence
        logger.info("\n" + "="*50)
        logger.info("ORIGINAL DISEASE PREVALENCE REPORT")
        logger.info("="*50)
        for disease in ['oestrus', 'calving', 'lameness', 'mastitis']:
            healthy = data[disease].value_counts().get(0, 0)
            diseased = data[disease].value_counts().get(1, 0)
            total = healthy + diseased
            logger.info(f"{disease.upper()}: {diseased} diseased ({diseased/total:.2%}), {healthy} healthy")
        
        # Calculate overall health status
        data['any_disease'] = data[['oestrus', 'calving', 'lameness', 'mastitis']].any(axis=1).astype(int)
        any_disease = data['any_disease'].sum()
        logger.info(f"\nOVERALL: {any_disease} animals with at least one disease ({any_disease/len(data):.2%})")
        logger.info(f"TOTAL RECORDS: {len(data)}")
        logger.info("="*50 + "\n")
        
        # Feature engineering
        data['rest_to_eat_ratio'] = np.where(
            (data['EAT'] == 0) & (data['REST'] == 0),
            1.0,
            np.where(
                data['EAT'] == 0,
                100.0,
                data['REST'] / data['EAT']
            )
        )
        
        # Cyclical time features
        hour_arr = data['hour'].values
        data['hour_sin'] = np.sin(2 * np.pi * hour_arr / 24)
        data['hour_cos'] = np.cos(2 * np.pi * hour_arr / 24)
        
        features = required_cols + ['rest_to_eat_ratio', 'hour_sin', 'hour_cos', 'any_disease']
        
        # Clean data
        clean_data = data.dropna(subset=features)
        if len(clean_data) < len(data):
            logger.warning(f"Removed {len(data)-len(clean_data)} records with missing values")
        
        logger.info(f"Final preprocessed data shape: {clean_data.shape}")
        return clean_data
    
    except Exception as e:
        logger.exception("Data loading/preprocessing failed")
        raise

def create_balanced_datasets(data):
    """Create balanced datasets with detailed reporting"""
    logger.info("Creating balanced datasets through undersampling")
    
    # 1. Create multi-output balanced dataset (any_disease)
    logger.info("\n" + "="*50)
    logger.info("CREATING MULTI-OUTPUT BALANCED DATASET")
    logger.info("="*50)
    
    # Separate features and targets
    X = data.drop(columns=['oestrus', 'calving', 'lameness', 'mastitis', 'cow', 'date', 'any_disease'])
    y = data['any_disease']
    
    # Undersample majority class
    rus = RandomUnderSampler(sampling_strategy='auto', random_state=42)
    X_res, y_res = rus.fit_resample(X, y)
    sampled_indices = rus.sample_indices_
    
    # Create balanced dataset
    balanced_data = pd.DataFrame(X_res, columns=X.columns)
    balanced_data['any_disease'] = y_res
    for col in ['cow', 'date', 'oestrus', 'calving', 'lameness', 'mastitis']:
        balanced_data[col] = data[col].iloc[sampled_indices].values
    
    # Save dataset
    output_path = os.path.join(OUTPUT_DIR, 'multi_output_balanced.csv')
    balanced_data.to_csv(output_path, index=False)
    
    # Report results
    logger.info(f"Saved multi-output balanced dataset with {len(balanced_data)} records")
    logger.info("Disease distribution after undersampling:")
    any_disease = balanced_data['any_disease'].sum()
    logger.info(f"Healthy: {len(balanced_data) - any_disease}")
    logger.info(f"With disease(s): {any_disease}")
    
    for disease in ['oestrus', 'calving', 'lameness', 'mastitis']:
        count = balanced_data[disease].sum()
        logger.info(f"- {disease}: {count} cases ({count/len(balanced_data):.2%})")
    
    # 2. Create disease-specific balanced datasets
    diseases = ['oestrus', 'calving', 'lameness', 'mastitis']
    
    for disease in diseases:
        logger.info("\n" + "="*50)
        logger.info(f"CREATING {disease.upper()}-BALANCED DATASET")
        logger.info("="*50)
        
        # Undersampling for specific disease
        rus = RandomUnderSampler(sampling_strategy='auto', random_state=42)
        X_res, y_res = rus.fit_resample(X, data[disease])
        sampled_indices = rus.sample_indices_
        
        # Create disease-specific dataset
        disease_data = pd.DataFrame(X_res, columns=X.columns)
        disease_data[disease] = y_res
        
        # Add metadata and other columns
        for col in ['cow', 'date'] + [d for d in diseases if d != disease]:
            disease_data[col] = data[col].iloc[sampled_indices].values
        
        # Add any_disease column
        disease_data['any_disease'] = disease_data[diseases].any(axis=1).astype(int)
        
        # Save dataset
        output_path = os.path.join(OUTPUT_DIR, f'{disease}_balanced.csv')
        disease_data.to_csv(output_path, index=False)
        
        # Report results
        logger.info(f"Saved {disease}-balanced dataset with {len(disease_data)} records")
        healthy = len(disease_data[disease_data[disease] == 0])
        diseased = len(disease_data[disease_data[disease] == 1])
        logger.info(f"- Healthy: {healthy}")
        logger.info(f"- Diseased: {diseased} ({diseased/len(disease_data):.2%})")
    
    logger.info("\n" + "="*50)
    logger.info("UNDERSAMPLING COMPLETE!")
    logger.info(f"All balanced datasets saved to '{OUTPUT_DIR}' directory")
    logger.info("="*50)

if __name__ == "__main__":
    try:
        logger.info("Starting undersampling process")
        data = load_and_preprocess()
        create_balanced_datasets(data)
        logger.info("Process completed successfully")
    except Exception as e:
        logger.exception("Undersampling process failed")

2025-07-09 01:15:09,628 - INFO - Starting undersampling process
2025-07-09 01:15:09,628 - INFO - Loading data from full_data_unhealthy_imputed.csv
2025-07-09 01:15:15,372 - INFO - 
2025-07-09 01:15:15,374 - INFO - ORIGINAL DISEASE PREVALENCE REPORT
2025-07-09 01:15:15,374 - INFO - ==================================================
2025-07-09 01:15:15,447 - INFO - OESTRUS: 7944 diseased (0.41%), 1927723 healthy
2025-07-09 01:15:15,526 - INFO - CALVING: 4296 diseased (0.22%), 1931371 healthy
2025-07-09 01:15:15,613 - INFO - LAMENESS: 3216 diseased (0.17%), 1932451 healthy
2025-07-09 01:15:15,685 - INFO - MASTITIS: 1056 diseased (0.05%), 1934611 healthy
2025-07-09 01:15:15,781 - INFO - 
OVERALL: 16440 animals with at least one disease (0.85%)
2025-07-09 01:15:15,781 - INFO - TOTAL RECORDS: 1935667
2025-07-09 01:15:15,781 - INFO - ==================================================

2025-07-09 01:15:16,875 - INFO - Final preprocessed data shape: (1935667, 19)
2025-07-09 01:15:16,917 - INFO 